In [ ]:
!pip install peft bitsandbytes unsloth

In [ ]:
import torch
import torch.nn as nn
from unsloth import FastLanguageModel
from transformers import AutoTokenizer
from datasets import load_dataset
from peft import LoraConfig, get_peft_model
from transformers import TrainingArguments, Trainer
from trl import SFTConfig, SFTTrainer

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

In [ ]:
model_name = "unsloth/Llama-3.2-1B-Instruct"
max_sequence_length = 2048
dtype = None

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(model_name, max_seq_length=max_sequence_length, dtype=dtype, load_in_4bit=True)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 8,
    lora_dropout = 0,
    lora_alpha = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing = True
)

In [ ]:
dataset = load_dataset("sahil2801/codeAlpaca-20k", split="train")

In [ ]:
split_dataset = dataset.train_test_split(test_size=0.1, seed=42)

train_dataset = split_dataset["train"]
test_dataset = split_dataset["test"]

In [ ]:
train_dataset, dataset

In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template="llama-3.1"
)

def formatting_func(examples):
    convos = []

    for instruction, input_text, output in zip(
        examples["instruction"],
        examples["input"],
        examples["output"]
    ):
        if input_text.strip():
            user_message = f"{instruction}\n\n{input_text}"
        else:
            user_message = instruction

        convos.append([
            {"role": "user", "content": user_message},
            {"role": "assistant", "content": output},
        ])

    texts = [
        tokenizer.apply_chat_template(
            convo,
            tokenize=False,
            add_generation_prompt=False,
        )
        for convo in convos
    ]

    return {"text": texts}

dataset = dataset.map(formatting_func, batched=True)

print(dataset[0]["text"])

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
args = SFTConfig(
    output_dir="./results",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    logging_steps=25,
    save_strategy="epoch",
    eval_strategy = "epoch",
    num_train_epochs = 1,
    learning_rate = 2e-5,
    fp16 = True,
    bf16 = False,
    push_to_hub=True
)

In [ ]:
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    args = args,
    train_dataset = train_dataset,
    eval_dataset=test_dataset,
    dataset_text_feild="text"
)

In [ ]:
trainer.train()

Save

In [ ]:
model.save_pretrained("Llama-3.2-1B-code-Instruct")
tokenizer.save_pretrained("Llama-3.2-1B-code-Instruct")

In [ ]:
model.push_to_hub("ciphermosaic/Llama-3.2-1B-code-Instruct")
tokenizer.push_to_hub("ciphermosaic/Llama-3.2-1B-code-Instruct")

Merging


In [ ]:
model.save_pretrained_merged("Llama-3.2-1B-code-Instruct-merged", 
                             tokenizer, 
                             save_method="merged_16bit")

In [ ]:
model.push_to_hub_merged(
    "ciphermosaic/Llama-3.2-1B-code-Instruct-merged",
    tokenizer,
    save_method="merged_16bit",
)

Convert to gguf

In [ ]:
model.save_pretrained_gguf(
    "Llama-3.2-1B-code-Instruct-GGUF",
    tokenizer,
    quantization_method="q4_k_m"
)

In [ ]:
model.push_to_hub_gguf(
    "ciphermosaic/Llama-3.2-1B-code-Instruct-GGUF",
    tokenizer,
    quantization_method="q4_k_m"
)